# Hint Eval Results Analysis

Loads all `results/*.jsonl` runs produced by `hint_eval.py`, then surfaces cases where the
model **took the hint** (`uptake == True`) and shows whether the hint source was
**verbalized** in the chain-of-thought (`think`) and/or the final answer text (`answer`),
including *where* in the text the mention occurs.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

RESULTS_DIR = Path("results")

## Load all result files

In [ ]:
def parse_tag(stem):
    """Recover (model, source, subset) from the '{model}__{source}__{subset}' filename stem."""
    model, source_us, subset = stem.split("__", 2)
    return model, source_us.replace("_", " "), subset


def load_results(results_dir=RESULTS_DIR):
    rows = []
    for path in sorted(results_dir.glob("*.jsonl")):
        model, source, subset = parse_tag(path.stem)
        with open(path) as f:
            for line in f:
                rec = json.loads(line)
                rec.update(model=model, source=source, subset=subset)
                rows.append(rec)
    return pd.DataFrame(rows)


def load_summaries(results_dir=RESULTS_DIR):
    rows = []
    for path in sorted(results_dir.glob("*.summary.json")):
        with open(path) as f:
            rows.append(json.load(f))
    return pd.DataFrame(rows)


df = load_results()
summaries = load_summaries()
print(f"Loaded {len(df)} records from {df[['model', 'source', 'subset']].drop_duplicates().shape[0]} run(s).")
df.head()

## Run summaries

In [ ]:
summaries

## Uptake cases

Rows where the model switched to the hinted option (`uptake == True`).

In [ ]:
uptake_df = df[df["uptake"]].copy()
print(f"{len(uptake_df)} / {len(df)} records show uptake "
      f"({len(uptake_df) / len(df):.1%})")
uptake_df[["model", "source", "subset", "idx", "gold", "baseline_answer",
           "hint_letter", "hinted_answer", "verbalized_think", "verbalized_answer"]]

## Locating hint mentions

`hint_eval.py`'s `verbalized_think` / `verbalized_answer` flags are booleans only. To see
*where* the hint source is mentioned, we re-split each hinted output into its `<think>` and
answer channels (same logic as `split_channels` in `hint_eval.py`) and highlight every
occurrence of a hint-source content word (or the literal word "hint").

In [ ]:
STOP_WORDS = {"a", "my", "the", "on", "of"}


def split_channels(text):
    """Mirrors hint_eval.split_channels: (thinking, answer_text)."""
    m = re.search(r"<think>(.*?)</think>", text, flags=re.S)
    if m:
        return m.group(1), text[m.end():]
    if "</think>" in text:
        pre, post = text.split("</think>", 1)
        return pre, post
    return "", text


def source_pattern(source):
    words = [w.lower() for w in re.findall(r"[A-Za-z]+", source) if w.lower() not in STOP_WORDS]
    words = sorted(set(words + ["hint"]), key=len, reverse=True)
    return re.compile(r"\b(" + "|".join(re.escape(w) for w in words) + r")\b", re.IGNORECASE)


def find_mentions(text, source, window=60):
    """Return highlighted context snippets for every hint-source mention in text.

    Nearby/overlapping matches are merged into a single snippet so a source like
    "a Stanford professor" (two content words next to each other) doesn\'t produce
    a near-duplicate snippet per word.
    """
    pattern = source_pattern(source)
    spans = [(max(0, m.start() - window), min(len(text), m.end() + window))
             for m in pattern.finditer(text)]
    if not spans:
        return []

    merged = [spans[0]]
    for start, end in spans[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))

    snippets = []
    for start, end in merged:
        snippet = text[start:end].replace("\n", " ")
        snippet = pattern.sub(lambda mm: f"**{mm.group(0)}**", snippet)
        prefix = "... " if start > 0 else ""
        suffix = " ..." if end < len(text) else ""
        snippets.append(f"{prefix}{snippet}{suffix}")
    return snippets

## Display uptake samples with verbalization locations

In [ ]:
def show_uptake_sample(row):
    think, answer_text = split_channels(row["hinted_output"])
    think_hits = find_mentions(think, row["source"]) if row["verbalized_think"] else []
    answer_hits = find_mentions(answer_text, row["source"]) if row["verbalized_answer"] else []

    model, source, subset = row["model"], row["source"], row["subset"]
    lines = [
        f"### [{model} | {source} | {subset}] idx={row['idx']}",
        f"**Question:** {row['question']}",
        "",
        f"- gold: `{row['gold']}`  baseline: `{row['baseline_answer']}`  "
        f"hint: `{row['hint_letter']}`  hinted answer: `{row['hinted_answer']}`",
        f"- verbalized in think: **{row['verbalized_think']}**  "
        f"verbalized in answer: **{row['verbalized_answer']}**",
    ]

    if think_hits:
        lines.append("\n**Hint mentioned in `think` channel:**")
        lines += [f"> {s}" for s in think_hits]
    if answer_hits:
        lines.append("\n**Hint mentioned in `answer` channel:**")
        lines += [f"> {s}" for s in answer_hits]
    if not think_hits and not answer_hits:
        lines.append("\n_Hint not verbalized in either channel._")

    lines.append("\n---")
    display(Markdown("\n\n".join(lines)))


for _, row in uptake_df.iterrows():
    show_uptake_sample(row)